
# Market Basket Analysis (Apriori) 
This notebook:
1. Loads a CSV of **user_id**s (first column).
2. Loads Instacart **PRIOR** data (`orders.csv`, `order_products__prior.csv`, `products.csv`) and filters to those users.
3. Builds transactions (one basket per prior order).
4. Runs **Apriori** to find frequent itemsets, then computes **support**, **confidence**, **lift**, and **interest** for association rules.
5. Includes helpers for querying rules and making basket-completion recommendations.

## Metrics refresher
Let A be an itemset (antecedent) and B a single item (consequent), N = number of baskets.
- **support(X)** = baskets containing X / N  
- **confidence(A→B)** = support(A ∪ {B}) / support(A)  
- **lift(A→B)** = confidence(A→B) / support(B) = support(A∪B)/(support(A)·support(B))  
- **interest(A→B)** = confidence(A→B) − support(B)  (additive improvement over baseline)


## 0) Setup & Paths

In [1]:

# %pip install pandas numpy mlxtend matplotlib
import os, itertools
import numpy as np
import pandas as pd

from collections import defaultdict
from typing import List

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option("display.max_colwidth", 120)

# --- EDIT THESE PATHS IF NEEDED ---
DATA_DIR = "data"          # folder with Instacart CSVs
USERS_FILE = "user_partitions.csv"  # CSV with user_id in the first column (header optional)

ORDERS_FILE = os.path.join(DATA_DIR, "orders.csv")
ORDER_LINES_FILE = os.path.join(DATA_DIR, "order_products__prior.csv")  # PRIOR for history
PRODUCTS_FILE = os.path.join(DATA_DIR, "products.csv")

print("Paths set.")


Paths set.


## 1) Load user list and PRIOR data

In [2]:
# Load user list (CSV has columns: user_id, partition)
users_df = pd.read_csv(USERS_FILE)

# Optional: filter a specific partition (uncomment and set the value you want)
# users_df = users_df[users_df["partition"] == "train"]  # e.g., "train" / "valid" / "test"

# Ensure we have a clean integer set of user_ids
if "user_id" not in users_df.columns:
    users_df = users_df.rename(columns={users_df.columns[0]: "user_id"})
user_series = pd.to_numeric(users_df["user_id"], errors="coerce").dropna().astype("int64")
USER_IDS = set(user_series.tolist())
print("Loaded user ids:", len(USER_IDS))

# Load Instacart data
orders = pd.read_csv(ORDERS_FILE, usecols=["order_id","user_id","order_number","days_since_prior_order"])
prior = pd.read_csv(ORDER_LINES_FILE, usecols=["order_id","product_id"])
products = pd.read_csv(PRODUCTS_FILE, usecols=["product_id","product_name"])

# Join product names and orders
data = (prior.merge(products, on="product_id", how="left")
              .merge(orders, on="order_id", how="left"))

# Filter to selected users
data_sel = data[data["user_id"].isin(USER_IDS)].copy()
num_orders = data_sel["order_id"].nunique()
num_items = data_sel["product_name"].nunique()
print("Selected users:", len(USER_IDS), "| prior orders kept:", num_orders, "| unique items:", num_items)

display(data_sel.head(3))


Loaded user ids: 100
Selected users: 100 | prior orders kept: 1721 | unique items: 4300


,order_id,product_id,product_name,user_id,order_number,days_since_prior_order
24723,2585,34358,Garlic,56048,36,7.0
24724,2585,5578,Okra,56048,36,7.0
24725,2585,2846,Organic Green Onions,56048,36,7.0


## 2) Build transactions (one basket per prior order)

In [3]:

baskets_df = data_sel.groupby("order_id")["product_name"].apply(lambda x: sorted(set(x))).reset_index()
transactions = baskets_df["product_name"].tolist()
N = len(transactions)
U = len(set(i for t in transactions for i in t))
print("Baskets:", N, "| Unique items:", U)


Baskets: 1721 | Unique items: 4300


## 3) Apriori: frequent itemsets and association rules

In [10]:
from collections import Counter
from pathlib import Path
from typing import Iterable, List, Optional, Tuple, Dict, Any

import numpy as np
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules


# -------------------------------------------------------------------
# 1) Frequent itemset mining (Apriori only, no rules here)
# -------------------------------------------------------------------
def mine_frequent_itemsets(
    transactions: List[Iterable],
    min_item_support: float = 0.001,
    support_grid: Optional[List[float]] = None,
    max_len: int = 3,
    use_low_memory: bool = True,
    verbose: bool = True,
) -> Tuple[pd.DataFrame, float]:
    """
    Run Apriori to mine frequent itemsets. This function does NOT
    create association rules; it only returns itemsets + chosen_support.

    Returns:
        freq: frequent itemsets DataFrame with 'itemsets' and 'support'
        chosen_support: the min_support actually used
    """
    # -------------------------
    # 0) Basic stats
    # -------------------------
    N = len(transactions)
    if N == 0:
        raise ValueError("No transactions provided")

    if verbose:
        print(f"[INFO] Transactions N={N}")

    # -------------------------
    # 1) Pre-filter rare items BEFORE one-hot encoding
    # -------------------------
    if min_item_support is not None and min_item_support > 0:
        abs_min_count = max(1, int(np.ceil(min_item_support * N)))
        item_counts = Counter()
        for t in transactions:
            item_counts.update(set(t))  # count once per basket

        keep_items = {item for item, c in item_counts.items() if c >= abs_min_count}
        if verbose:
            U_raw = len(item_counts)
            U_kept = len(keep_items)
            print(
                f"[INFO] Unique items before filtering: {U_raw} "
                f"-> after min_item_support={min_item_support:.4g}: {U_kept}"
            )

        filtered_transactions = [
            [item for item in t if item in keep_items]
            for t in transactions
        ]
    else:
        filtered_transactions = transactions

    # -------------------------
    # 2) TransactionEncoder -> dense bool DataFrame
    # -------------------------
    te = TransactionEncoder()
    X_bool = te.fit(filtered_transactions).transform(filtered_transactions)
    X = pd.DataFrame(X_bool, columns=te.columns_, dtype=bool)
    del X_bool

    U = X.shape[1]
    if verbose:
        print(f"[INFO] After filtering & encoding: N={N}, U={U} (bool matrix)")

    # -------------------------
    # 3) Support ladder
    #    Use the *lowest* support that yields any itemsets
    # -------------------------
    if support_grid is None:
        base = [0.05, 0.03, 0.02, 0.01, 0.005, 0.002, 0.001, max(1.0 / N, 0.0005)]
        support_grid = sorted({s for s in base if s > 0}, reverse=True)
    else:
        support_grid = sorted({s for s in support_grid if s > 0}, reverse=True)

    if verbose:
        print(f"[INFO] Support grid (high -> low): {support_grid}")

    freq = pd.DataFrame()
    chosen_support = None

    last_nonempty = None
    last_s = None

    for s in support_grid:
        f = apriori(
            X,
            min_support=s,
            use_colnames=True,
            max_len=max_len,
            low_memory=use_low_memory,
        )

        lens = f["itemsets"].apply(len)
        pairs = f[lens == 2]

        if verbose:
            print(
                f"[DBG] min_support={s:.4f} -> "
                f"itemsets={len(f)} (pairs={len(pairs)})"
            )

        if len(f) > 0:
            last_nonempty = f
            last_s = s

        del f, pairs

    if last_nonempty is not None:
        freq = last_nonempty.sort_values("support", ascending=False).reset_index(drop=True)
        chosen_support = last_s

    if freq.empty or chosen_support is None:
        raise ValueError(
            "No frequent itemsets found even at lowest support. "
            "Try lowering min_item_support or adding more baskets."
        )

    lens = freq["itemsets"].apply(len)
    n1 = (lens == 1).sum()
    n2 = (lens == 2).sum()
    n3 = (lens == 3).sum()
    if verbose:
        print(
            f"[INFO] Using min_support={chosen_support:.4f}. "
            f"Singles={n1}, Pairs={n2}, Triples={n3}"
        )

    return freq, chosen_support


# -------------------------------------------------------------------
# 2) Association rule generation (fully configurable)
# -------------------------------------------------------------------
def build_association_rules(
    freq: pd.DataFrame,
    chosen_support: Optional[float] = None,
    *,
    # core association_rules parameters
    rule_metric: str = "confidence",
    rule_min_threshold: float = 0.1,
    # strict pruning
    min_lift: Optional[float] = 1.0,
    min_confidence: Optional[float] = None,
    min_interest: Optional[float] = None,
    prune_by_support: bool = True,
    # soft mode
    min_rules: Optional[int] = None,
    soft_min_confidence: float = 0.02,
    soft_min_lift: float = 1.0,
    soft_prune_by_support: bool = False,
    # global cap
    max_rules: Optional[int] = None,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Build and prune association rules from frequent itemsets.

    This is the function you can easily tweak or wrap if you want
    different rule behavior, without re-running Apriori.

    Returns:
        selected: pruned/ordered association rules DataFrame
    """
    # -------------------------
    # 4) Association rules (ALL)
    # -------------------------
    rules_all = association_rules(
        freq, metric=rule_metric, min_threshold=rule_min_threshold
    ).copy()

    if verbose:
        print(f"[INFO] Raw rules: {len(rules_all)}")

    if rules_all.empty:
        if verbose:
            print(
                "[HINT] No rules produced. You need at least some 2-itemsets; "
                "try lowering min_support or adjusting rule_min_threshold."
            )
        return rules_all

    # -------------------------
    # 5) interest = confidence - support(consequent)
    # -------------------------
    lens = freq["itemsets"].apply(len)
    singles = freq[lens == 1][["itemsets", "support"]]
    conseq_support = {
        next(iter(s)): sup for s, sup in zip(singles["itemsets"], singles["support"])
    }

    def get_consequent_support(cs: frozenset) -> float:
        return conseq_support.get(next(iter(cs)), np.nan)

    rules_all["interest"] = rules_all["confidence"] - rules_all["consequents"].apply(
        get_consequent_support
    )

    # -------------------------
    # 6) Strict pruning
    # -------------------------
    strict_mask = pd.Series(True, index=rules_all.index)

    if min_lift is not None:
        strict_mask &= rules_all["lift"] >= min_lift
    if prune_by_support and chosen_support is not None:
        strict_mask &= rules_all["support"] >= chosen_support
    if min_confidence is not None:
        strict_mask &= rules_all["confidence"] >= min_confidence
    if min_interest is not None:
        strict_mask &= rules_all["interest"] >= min_interest

    strict_rules = rules_all[strict_mask].copy()

    if verbose:
        print(f"[INFO] Strict pruning -> {len(strict_rules)} rules")

    # -------------------------
    # 7) Soft pruning fallback
    # -------------------------
    if min_rules is None or min_rules <= 0:
        selected = strict_rules
        mode = "strict-only"
    else:
        if len(strict_rules) >= min_rules:
            selected = strict_rules
            mode = "strict"
        else:
            soft_mask = pd.Series(True, index=rules_all.index)

            if soft_min_lift is not None:
                soft_mask &= rules_all["lift"] >= soft_min_lift
            if soft_min_confidence is not None:
                soft_mask &= rules_all["confidence"] >= soft_min_confidence
            if soft_prune_by_support and chosen_support is not None:
                soft_mask &= rules_all["support"] >= chosen_support

            soft_rules = rules_all[soft_mask].copy()

            if verbose:
                print(
                    f"[INFO] Strict rules ({len(strict_rules)}) < min_rules={min_rules}, "
                    f"using soft pruning -> {len(soft_rules)} rules"
                )

            selected = soft_rules
            mode = "soft"

    selected = selected.sort_values(
        ["lift", "confidence", "support"], ascending=False
    ).reset_index(drop=True)

    if max_rules is not None and max_rules > 0:
        selected = selected.head(max_rules)

    if verbose:
        print(
            f"[INFO] After {mode} pruning & capping: {len(selected)} rules "
            f"(max_rules={max_rules}, min_rules={min_rules})"
        )

    return selected


# -------------------------------------------------------------------
# 3) Optional wrapper to keep your original API
# -------------------------------------------------------------------
def market_basket_apriori(
    transactions: List[Iterable],
    # item filtering / encoding
    min_item_support: float = 0.001,
    # frequent itemset search
    support_grid: Optional[List[float]] = None,
    max_len: int = 3,
    use_low_memory: bool = True,
    # rule generation (strict thresholds)
    rule_metric: str = "confidence",
    rule_min_threshold: float = 0.1,
    min_lift: Optional[float] = 1.0,
    min_confidence: Optional[float] = None,
    min_interest: Optional[float] = None,
    prune_by_support: bool = True,
    # soft mode
    min_rules: Optional[int] = None,
    soft_min_confidence: float = 0.02,
    soft_min_lift: float = 1.0,
    soft_prune_by_support: bool = False,
    # global cap & I/O & logging
    max_rules: Optional[int] = None,
    out_dir: Optional[str] = None,
    out_prefix: str = "selected_users",
    verbose: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame, float]:
    """
    Backwards-compatible wrapper:
    - mines itemsets
    - builds rules
    - saves outputs (optional)
    """
    freq, chosen_support = mine_frequent_itemsets(
        transactions=transactions,
        min_item_support=min_item_support,
        support_grid=support_grid,
        max_len=max_len,
        use_low_memory=use_low_memory,
        verbose=verbose,
    )

    rules = build_association_rules(
        freq=freq,
        chosen_support=chosen_support,
        rule_metric=rule_metric,
        rule_min_threshold=rule_min_threshold,
        min_lift=min_lift,
        min_confidence=min_confidence,
        min_interest=min_interest,
        prune_by_support=prune_by_support,
        min_rules=min_rules,
        soft_min_confidence=soft_min_confidence,
        soft_min_lift=soft_min_lift,
        soft_prune_by_support=soft_prune_by_support,
        max_rules=max_rules,
        verbose=verbose,
    )

    _save_outputs(freq, rules, out_dir, out_prefix, verbose)
    return freq, rules, chosen_support


def _save_outputs(
    freq: pd.DataFrame,
    rules: pd.DataFrame,
    out_dir: Optional[str],
    out_prefix: str,
    verbose: bool,
):
    if out_dir is None:
        return
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    freq_out = Path(out_dir) / f"{out_prefix}_itemsets_apriori.csv"
    rules_out = Path(out_dir) / f"{out_prefix}_rules_apriori.csv"

    freq.to_csv(freq_out, index=False)
    rules.to_csv(rules_out, index=False)

    if verbose:
        print(f"[INFO] Saved itemsets to: {freq_out}")
        print(f"[INFO] Saved rules to:   {rules_out}")


# -------------------------------------------------------------------
# 4) Example usage (very close to your original)
# -------------------------------------------------------------------
freq, rules, chosen_support = market_basket_apriori(
    transactions=transactions,
    min_item_support=0.001,
    support_grid=[0.01],
    max_len=5,
    rule_metric="confidence",
    rule_min_threshold=0.01,
    min_lift=None,
    min_confidence=None,
    min_interest=None,
    prune_by_support=False,
    min_rules=5,
    soft_min_confidence=0.02,
    soft_min_lift=1.0,
    soft_prune_by_support=False,
    max_rules=800000,
    out_dir="data",
    out_prefix="selected_users",
    verbose=True,
)

print(f"[DBG] rules rows: {len(rules)}")

# 5) Build rules_idx AFTER we have `rules`
rules_idx = []
for _, r in rules.iterrows():
    A = tuple(sorted(list(r["antecedents"])))
    C = list(r["consequents"])[0] if len(r["consequents"]) == 1 else None
    if C is None:
        continue
    rules_idx.append((A, C, r["lift"], r["confidence"], r["support"], r["interest"]))

print(f"[DBG] rules_idx entries: {len(rules_idx)}")


[INFO] Transactions N=1721
[INFO] Unique items before filtering: 4300 -> after min_item_support=0.001: 2350
[INFO] After filtering & encoding: N=1721, U=2350 (bool matrix)
[INFO] Support grid (high -> low): [0.01]
[DBG] min_support=0.0100 -> itemsets=247 (pairs=47)
[INFO] Using min_support=0.0100. Singles=200, Pairs=47, Triples=0
[INFO] Raw rules: 94
[INFO] Strict pruning -> 94 rules
[INFO] After strict pruning & capping: 94 rules (max_rules=800000, min_rules=5)
[INFO] Saved itemsets to: data/selected_users_itemsets_apriori.csv
[INFO] Saved rules to:   data/selected_users_rules_apriori.csv
[DBG] rules rows: 94
[DBG] rules_idx entries: 94


## 4) Helpers — query rules and recommend items

In [12]:

# Index rules (only single-item consequents)
rules_idx = []
for _, r in rules.iterrows():
    A = tuple(sorted(list(r["antecedents"])))
    C = list(r["consequents"])[0] if len(r["consequents"])==1 else None
    if C is None:
        continue
    rules_idx.append((A, C, r["lift"], r["confidence"], r["support"], r["interest"]))

def what_goes_with(item: str, top_k: int=10) -> pd.DataFrame:
    sub = [row for row in rules_idx if row[1] == item]
    if not sub:
        return pd.DataFrame(columns=["antecedent","consequent","support","confidence","lift","interest"])
    sub_sorted = sorted(sub, key=lambda x: (x[2], x[3], x[4]), reverse=True)[:top_k]
    return pd.DataFrame([(a,(c),s,conf,lft,intt) for a,c,lft,conf,s,intt in sub_sorted],
                        columns=["antecedent","consequent","support","confidence","lift","interest"])

def recommend_for_basket(current_items: List[str], k: int=10, subset_k: int=2) -> pd.DataFrame:
    S = sorted(set(current_items))
    best = defaultdict(lambda: 0.0)
    meta = {}
    for A, C, lift_, conf_, supp_, intr_ in rules_idx:
        if len(A) <= subset_k and set(A).issubset(S):
            score = lift_ * conf_
            if C not in S and score > best[C]:
                best[C] = score
                meta[C] = {"antecedent": A, "lift": lift_, "confidence": conf_, "support": supp_, "interest": intr_}
    rows = [(i, s, meta[i]["antecedent"], meta[i]["lift"], meta[i]["confidence"], meta[i]["support"], meta[i]["interest"]) 
            for i, s in best.items()]
    recs = pd.DataFrame(rows, columns=["item","score","because","lift","confidence","support","interest"]).sort_values("score", ascending=False).head(k)
    return recs

print(f"[DBG] rules rows: {len(rules)}")
print(f"[DBG] rules_idx entries: {len(rules_idx)}")

# Demo basket: top-3 most frequent items among selected users (fall back to 1–2 if needed)
popular = (data_sel.groupby("product_name")
                    .size()
                    .sort_values(ascending=False)
                    .head(3)
                    .index.tolist())
if not popular:
    # fallback: any items at all?
    popular = list(data_sel["product_name"].value_counts().head(3).index)
print("[DBG] Demo basket:", popular)

# Try with subset_k up to 3 in case your rules use larger antecedents
recs = recommend_for_basket(popular, k=10, subset_k=3) if popular else pd.DataFrame()

if recs is None or recs.empty:
    # Explain why nothing showed
    note_rows = []
    if len(rules_idx) == 0:
        note_rows.append("No rules available. Lower min_support/confidence or remove extra pruning.")
    if not popular:
        note_rows.append("Demo basket is empty—check data_sel or pick a manual basket like ['Milk','Bread'].")
    if not note_rows:
        note_rows.append("No rules matched this basket. Try increasing subset_k or use a different basket.")
    recs = pd.DataFrame({"note": note_rows})
    print("[INFO] No recommendations to display; see notes below.")

display(recs)


[DBG] rules rows: 94
[DBG] rules_idx entries: 94
[DBG] Demo basket: ['Bag of Organic Bananas', 'Banana', 'Organic Strawberries']


,item,score,because,lift,confidence,support,interest
0,Plain Yogurt,1.687037,"(Banana,)",11.487921,0.146853,0.012202,0.134070
1,Original Puffins Cereal,1.519099,"(Banana,)",11.433217,0.132867,0.011040,0.121246
2,Bartlett Pears,1.323083,"(Banana,)",7.276956,0.181818,0.015107,0.156833
3,100% Whole Wheat Bread,1.127361,"(Banana,)",6.717190,0.167832,0.013945,0.142847
5,Cucumber Kirby,1.015106,"(Banana,)",5.184293,0.195804,0.016270,0.158035
6,Half & Half,0.824462,"(Banana,)",5.126004,0.160839,0.013364,0.129462
4,Organic Baby Broccoli,0.769979,"(Bag of Organic Bananas,)",6.646138,0.115854,0.011040,0.098422
7,Hass Avocados,0.632958,"(Banana,)",4.763840,0.132867,0.011040,0.104976
9,Organic Raspberries,0.629528,"(Organic Strawberries,)",3.657256,0.172131,0.012202,0.125065
8,Asparagus,0.462170,"(Banana,)",3.671684,0.125874,0.010459,0.091592
